In [2]:
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame
from alpaca.trading.client import TradingClient
from alpaca.trading.requests import MarketOrderRequest
from alpaca.trading.enums import OrderSide, TimeInForce
import time
import datetime
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

API_KEY = "PKPZZGSS6633WRSOZJVRZ5Q6QX"
SECRET_KEY = "FL71bYjFTdmesWxs7i4Sw3aEzyDMArTYPpsHvK5E7zGF"

client = StockHistoricalDataClient(API_KEY, SECRET_KEY)

In [3]:
def fetch_bars(client, symbol, start, end, tf_override):
    request = StockBarsRequest(
        symbol_or_symbols=[symbol],
        timeframe=TimeFrame.Minute,
        start=start,
        end=end,
        timeframe_override=tf_override
    )
    bars = client.get_stock_bars(request)[symbol]
    df = pd.DataFrame([{
        "timestamp": bar.timestamp,
        "open": bar.open,
        "high": bar.high,
        "low": bar.low,
        "close": bar.close,
        "volume": bar.volume,
        "trade_count": bar.trade_count
    } for bar in bars])
    df.set_index("timestamp", inplace=True)
    df.sort_index(inplace=True)
    return df
start_date = '2015-01-01'
end_date = '2025-01-01'

data_5min = fetch_bars(client, "SPY", start_date, end_date, tf_override=5)
data = fetch_bars(client, "SPY", start_date, end_date, tf_override=15)

KeyboardInterrupt: 

In [ ]:
data['ema_100'] = data['close'].ewm(span=100, adjust=False).mean()
data['ema_200'] = data['close'].ewm(span=200, adjust=False).mean()

'''
ATR can be defined as EMA (max(high - low, abs(high - prev_close), abs(low - prev_close))) over a period usually equal to 14
'''
#Setting up ATR
def calc_ATR(df, period = 14):
  high = df['high']
  low = df['low']
  close = df['close']
  prev_close = close.shift(1)
  tr1 = high - low
  tr2 = abs(high - prev_close)
  tr3 = abs(low - prev_close)
  tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
  atr = tr.ewm(span=period, adjust=False).mean()
  return atr

data['ATR'] = calc_ATR(data)

'''
delta = close - close.shift(1)
Separate gain and loss
Average gain/loss over n periods (usually 14)
RS = avg_gain / avg_loss
RSI = 100 - (100 / (1 + RS))
'''
#Setting up RSI (Stochastic)
def calc_RSI(df, period = 14):
  delta = df['close'].diff()
  gain = delta.clip(lower=0)
  loss = delta.clip(upper=0)
  avg_gain = gain.ewm(span=period, adjust=False).mean()
  avg_loss = loss.ewm(span=period, adjust=False).mean()
  rs = avg_gain/avg_loss
  rsi = 100 - (100 / (1 + rs))
  mn = rsi.rolling(14).min()
  mx = rsi.rolling(14).max()
  stoch_rsi = (rsi - mn) / (mx - mn)
  k_rsi = stoch_rsi.rolling(3).mean()
  d_rsi = stoch_rsi.rolling(5).mean()
  return k_rsi

data['RSI'] = calc_RSI(data)

In [ ]:
def compute_trend_ratio():
    short_avg = data['close'].resample('15min').mean()
    long_avg = data['close']

    # Align the two time series
    combined = pd.DataFrame({
        'short_avg': short_avg,
        'long_avg': long_avg
    }).dropna()

    # Compute the trend ratio
    combined['trend_ratio'] = combined['short_avg'] / combined['long_avg']

    return combined[['trend_ratio']]
data['trend_ratio'] = compute_trend_ratio()

In [ ]:
#Initialising signals
data['signal'] = 0
#Setting up long and short conditions

#long condition
long_cond = ((data['close'] > data['high'].shift(1) + 0.5 * data['ATR']) &
             (data['ema_100'] > data['ema_200']) &
             (data['RSI'] > 0.8) &
             (data['close'] > data['ema_200'])#&
             #(data['trend_ratio'] > 1.0)
             )

#short condition
short_cond = ((data['close'] < data['low'].shift(1) - 0.5 * data['ATR']) &
              (data['ema_100'] < data['ema_200']) &
              (data['RSI'] < 0.2) &
              (data['close'] < data['ema_200'])#&
              #(data['trend_ratio'] < 1.0)
              )

data.loc[long_cond, 'signal'] = -1
data.loc[short_cond & (~long_cond), 'signal'] = 1

In [ ]:
'''
Implementing a stop such that if n continous signals of the same kind have been generated then wait till a signal of the opposite kind has been generated to start generating signals again.
'''
def throttle_signals(signal_series, max_repeats=3):
    filtered = []
    last_signal = 0
    same_signal_count = 0
    throttling = False

    for sig in signal_series:
        if sig == 0:
            filtered.append(0)
            continue

        if sig == last_signal:
            same_signal_count += 1
        else:
            same_signal_count = 1
            throttling = False

        if same_signal_count > max_repeats:
            throttling = True

        if throttling:
            filtered.append(0)
        else:
            filtered.append(sig)

        last_signal = sig if sig != 0 else last_signal

    return pd.Series(filtered, index=signal_series.index)
'''
To set it as buy only:
        if signal == 1:
            position = 1
        elif signal == -1:
            position = 0
        elif signal == 0:
            pass  # maintain current position
To set it as sell only:
        if signal == 1:
            position = 0
        elif signal == -1:
            position = -1
        elif signal == 0:
To set it as buy and sell:
        if signal == 1:
            position = 1
        elif signal == -1:
            position = -1
        elif signal == 0:
The results for all three are attached as images at the end of this file.
'''
'''
for sell only according to kelly leveraging, almost all trades are best and it is best not to use this logic. the results without kelly leveraging have been added in the bottom as well.
'''
def generate_positions(signal_series):
    position = 0
    positions = []

    for signal in signal_series:
        if signal == 1:
            position = 0
        elif signal == -1:
            position = -1
        elif signal == 0:
            pass  # maintain current position

        positions.append(position)

    return pd.Series(positions, index=signal_series.index)


data['positions'] = generate_positions(throttle_signals(data['signal'], max_repeats=1))


In [ ]:
strat_col = 'slippage_returns'


def CAGR(df):
    total_return = (1 + df[strat_col]).prod()
    total_seconds = (df.index[-1] - df.index[0]).total_seconds()
    years = total_seconds / (365.25 * 24 * 60 * 60)

    return (total_return ** (1 / years)) - 1

def volatility(df, bars_per_year=252 * 26, strat_col='slippage_returns'):
    return df[strat_col].std() * np.sqrt(bars_per_year)


def sharpe_ratio(df, risk_free_rate=0.05):
    vol = volatility(df)
    if vol == 0:
        return np.nan
    return (CAGR(df) - risk_free_rate) / vol


def max_drawdown(df):
    cum = (1 + df[strat_col]).cumprod()
    peak = cum.cummax()
    drawdown = (cum - peak) / peak
    return drawdown.min()

def win_loss_ratio(df):

    # Identify entry rows
    entries = df[df['positions'].diff().fillna(0) != 0].index
    returns = df.loc[entries, strat_col]
    wins = returns[returns > 0].count()
    losses = returns[returns < 0].count()
    total = wins + losses
    return (wins / total * 100 if total else 0,
            losses / total * 100 if total else 0)

# Daily returns
data['returns'] = data['close'].pct_change()

# Strategy returns based on positions
data['strategy_returns'] = data['returns'] * data['positions'].shift(1)

# Kelly fraction
kelly_fraction = data['strategy_returns'].mean() / data['strategy_returns'].var()
kelly_fraction = float(np.clip(kelly_fraction, 0, 2))
# Apply Kelly leverage
data['kelly_returns'] = data['strategy_returns'] #* kelly_fraction

# Slippage adjusted
slippage = 0.001
data['slippage_returns'] = data['kelly_returns'] * (1 - slippage)


# Compute KPIs
cagr = CAGR(data)
vol = volatility(data)
sharpe = sharpe_ratio(data)
mdd = max_drawdown(data)
win_pct, loss_pct = win_loss_ratio(data)
total_return = (1 + data[strat_col]).prod() - 1

# Print results
print("=== S&P500 Strategy Performance===")
print(f"CAGR: {cagr:.2%}")
print(f"Annualized Volatility: {vol:.2%}")
print(f"Sharpe Ratio (5% RF): {sharpe:.2f}")
print(f"Maximum Drawdown: {mdd:.2%}")
print(f"Win %: {win_pct:.2f}%")
print(f"Loss %: {loss_pct:.2f}%")
print(f"Total Return: {total_return:.2%}")
print('\n')
print("Total Buy signals:", (data['positions'] == 1).sum())
print("Total Sell signals:", (data['positions'] == -1).sum())
exposure_ratio = (data['signal'] != 0).mean()
print(f"Market exposure: {exposure_ratio * 100:.2f}%")
avg_return = data['returns'].mean()
print(f"Avg trade return: {avg_return:.4%}")
print('\n')


total_return1 = (1 + data['returns']).prod()
total_seconds1 = (data.index[-1] - data.index[0]).total_seconds()
years1 = total_seconds1 / (365.25 * 24 * 60 * 60)
print('Sharpe of buy and hold')
print((total_return1 ** (1 / years1)) - 1)

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(data.index, data[('close')], label='Close Price', color='black')

# Entry points
buy_entries = data[throttle_signals(data['signal'], max_repeats=1)==1].index
sell_entries = data[throttle_signals(data['signal'], max_repeats=1)==-1].index

# Plot buys
plt.plot(buy_entries, data.loc[buy_entries, ('close')],
         '^', color='green', label='Buy Entry', markersize=12)

# Plot sells
plt.plot(sell_entries, data.loc[sell_entries, ('close')],
         'v', color='red', label='Sell Entry', markersize=12)

plt.title('Entry and Exit Points (Using positions)')
plt.legend()
plt.grid(True)
plt.show()



# Calculate cumulative returns
cumulative_returns = (data['kelly_returns'].fillna(0) + 1).cumprod()
cumulative_slippage = (data['slippage_returns'].fillna(0) + 1).cumprod()
cumulative_benchmark = (data['returns'].fillna(0) + 1).cumprod()

# Create a single figure and plot all three lines
plt.figure(figsize=(12, 6))

plt.plot(cumulative_returns, label='strategy_returns', color='blue')
plt.plot(cumulative_slippage, label='slippage_returns', color='orange')
plt.plot(cumulative_benchmark, label='returns', color='green')

plt.title('Cumulative Strategy Returns')
plt.xlabel('Date')
plt.ylabel('Cumulative Return')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()